In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.config import DATASET_FILE, METADATA_FILE

df   = pd.read_csv(DATASET_FILE)
meta = pd.read_csv(METADATA_FILE)

print(f"Countries : {len(df)}")
print(f"Criteria  : {len(meta)}")
df.head()

Countries : 26
Criteria  : 8


,Country,Employment rate,Long-term unemployment rate,Personal earnings,Life expectancy,Life satisfaction,Employees working very long hours,Air pollution,Distance from Poznan (km)
0,Austria,72.0,1.3,53132.0,82.0,7.2,5.3,12.2,468.5
1,Belgium,65.0,2.3,54327.0,82.1,6.8,4.3,12.8,883.8
2,Czech Republic,74.0,0.6,29885.0,79.3,6.9,4.5,17.0,311.7
3,Denmark,74.0,0.9,58430.0,81.5,7.5,1.1,10.0,461.5
4,Estonia,74.0,1.2,30720.0,78.8,6.5,2.2,5.9,920.1


In [2]:
from ahp.hierarchy_setup import CATEGORIES, CATEGORY_CRITERIA, CRITERION_CATEGORY, CRITERIA, DIRECTIONS

print(CRITERIA)
print(DIRECTIONS)

['Employment rate', 'Long-term unemployment rate', 'Personal earnings', 'Life expectancy', 'Life satisfaction', 'Employees working very long hours', 'Air pollution', 'Distance from Poznan (km)']
{'Employment rate': 1, 'Long-term unemployment rate': -1, 'Personal earnings': 1, 'Life expectancy': 1, 'Life satisfaction': 1, 'Employees working very long hours': -1, 'Air pollution': -1, 'Distance from Poznan (km)': -1}


In [3]:
from ahp.weights import ahp_weights

A_test = np.array([
    [1,   3],
    [1/3, 1],
])

result = ahp_weights(A_test)
print(result["weights"])   # should be roughly [0.75, 0.25]
print(result["lambda_max"])
# should be exactly 2.0 for a 2x2 consistent matrix
print(f"CR = {result['CR']:.4f}")
print(f"Consistent: {result['consistent']}")

[0.75 0.25]
2.0
CR = 0.0000
Consistent: True


## AHP Hierarchy and Pairwise Comparison Matrices

The decision problem is structured as a 3-level hierarchy:

**Goal:** Find the best European country to live in for a student from Poznań.
```
GOAL
├── Economic
│       ├── Employment rate
│       ├── Long-term unemployment rate
│       └── Personal earnings
├── Social/Health
│       ├── Life expectancy
│       ├── Life satisfaction
│       └── Employees working very long hours
└── Geography/Environment
        ├── Air pollution
        └── Distance from Poznan (km)
```

At each level the DM provides pairwise comparisons using Saaty's 1-9 scale:

| Score | Meaning |
|-------|---------|
| 1 | Equal importance |
| 3 | Moderate importance |
| 5 | Strong importance |
| 7 | Very strong importance |
| 9 | Extreme importance |

**DM judgments (Goal level):**
- Economic is **strongly** more important than Social/Health (5×)
- Economic is **moderately** more important than Geography/Environment (3×)
- Social/Health is **slightly** more important than Geography/Environment (2×)

Note: the Goal matrix is intentionally inconsistent (CR=0.14 > 0.10) —
the three judgments are not perfectly transitive, which is natural in human judgment.

In [4]:
from ahp.dm_matrices import MATRICES
from ahp.weights import ahp_weights

for name, A in MATRICES.items():
    r = ahp_weights(A)
    mark = "Satisfactory" if r["consistent"] else "Inconsistent"
    print(f"{name:<28s}  CR={r['CR']:.4f}  {mark}")

Goal                          CR=0.1407  Inconsistent
Economic                      CR=0.0462  Satisfactory
Social/Health                 CR=0.0079  Satisfactory
Geography/Environment         CR=0.0000  Satisfactory


In [5]:
from ahp.global_weights import compute_global_weights
from ahp.dm_matrices import MATRICES
from ahp.hierarchy_setup import CATEGORIES

results = {name: ahp_weights(A) for name, A in MATRICES.items()}

w_goal = results["Goal"]["weights"]
w_cats = {cat: results[cat]["weights"] for cat in CATEGORIES}

global_weights = compute_global_weights(w_goal, w_cats)

for crit, w in sorted(global_weights.items(), key=lambda x: -x[1]):
    print(f"{crit:<40s}  {w:.4f}")

Personal earnings                         0.4658
Long-term unemployment rate               0.1174
Distance from Poznan (km)                 0.1100
Life satisfaction                         0.1059
Employment rate                           0.0739
Employees working very long hours         0.0583
Air pollution                             0.0367
Life expectancy                           0.0321


In [6]:
from ahp.consistency import reconstruct_matrix, max_discrepancy
from ahp.hierarchy_setup import CATEGORIES

A_orig = MATRICES["Goal"]
A_rec  = reconstruct_matrix(results["Goal"]["weights"])
disc   = max_discrepancy(A_orig, A_rec, CATEGORIES)

print(f"Biggest discrepancy in Goal matrix:")
print(f"  {disc['row_label']} vs {disc['col_label']}")
print(f"  DM value     : {disc['dm_value']:.4f}")
print(f"  Reconstructed: {disc['rec_value']:.4f}")
print(f"  Difference   : {disc['diff']:.4f}")

Biggest discrepancy in Goal matrix:
  Economic vs Social/Health
  DM value     : 5.0000
  Reconstructed: 3.3472
  Difference   : 1.6528


In [7]:
from ahp.alternative_matrices import diff_to_score, THRESHOLDS

# Denmark earnings: 58430, Greece earnings: 27207
# difference = 31223 -> should map to score 8
diff = abs(58430 - 27207)
score = diff_to_score(diff, THRESHOLDS["Personal earnings"])
print(f"Difference: {diff}, Saaty score: {score}")

Difference: 31223, Saaty score: 8


In [8]:
from ahp.alternative_matrices import build_alternative_matrix

A = build_alternative_matrix("Personal earnings", df)
print(A.shape)       # should be (26, 26)
print(A[0, 1])       # Austria vs Belgium on earnings
print(A[1, 0])       # should be reciprocal of above

(26, 26)
1.0
1.0


In [9]:
# Switzerland (highest earnings) vs Greece (lowest)
ch_idx = df[df["Country"] == "Switzerland"].index[0]
gr_idx = df[df["Country"] == "Greece"].index[0]

print(f"Switzerland earnings: {df.loc[ch_idx, 'Personal earnings']}")
print(f"Greece earnings: {df.loc[gr_idx, 'Personal earnings']}")
print(f"A[CH, GR] = {A[ch_idx, gr_idx]}")   # should be 9
print(f"A[GR, CH] = {A[gr_idx, ch_idx]}")   # should be 1/9

Switzerland earnings: 64824.0
Greece earnings: 27207.0
A[CH, GR] = 9.0
A[GR, CH] = 0.1111111111111111


In [10]:
from ahp.scoring import compute_ahp_scores
import pandas as pd

scores = compute_ahp_scores(df, global_weights)

ranking = pd.DataFrame({
    "Country":   df["Country"].values,
    "AHP Score": scores,
}).sort_values("AHP Score", ascending=False).reset_index(drop=True)

ranking.index += 1
ranking.index.name = "Rank"
print(ranking)

              Country  AHP Score
Rank                            
1             Iceland   0.090508
2         Switzerland   0.088341
3          Luxembourg   0.077722
4         Netherlands   0.069335
5             Denmark   0.067769
6              Norway   0.055017
7             Germany   0.054431
8             Finland   0.046423
9             Austria   0.045818
10             Sweden   0.043187
11            Belgium   0.035973
12            Ireland   0.034122
13             Poland   0.031855
14     Czech Republic   0.030574
15     United Kingdom   0.030297
16            Estonia   0.024042
17             France   0.022690
18           Slovenia   0.021662
19          Lithuania   0.021550
20            Hungary   0.020718
21             Latvia   0.018473
22              Spain   0.017729
23              Italy   0.016349
24    Slovak Republic   0.015584
25           Portugal   0.012070
26             Greece   0.007764
